# Speech Emotion Recognition on CREMA-D

This notebook is the streamlined entry point for experimenting with speech emotion recognition on the CREMA-D dataset. It supports three training modes and both classical and quantum heads:

- **Transfer learning on precomputed embeddings** (frozen features + small head)
- **Head-only fine-tuning** using a pretrained backbone + embedding stack
- **CNN trained from scratch** on MEL or MFCC tensors

Use the controls below to pick the experiment type, classical vs quantum, and the core hyperparameters. The code relies on the modular utilities under `src/` for datasets, models, and training.


In [ ]:

import os
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from ipywidgets import (
    Button,
    Dropdown,
    FloatText,
    HBox,
    IntSlider,
    Layout,
    Output,
    ToggleButtons,
    VBox,
)

# Ensure src/ is importable
ROOT_DIR = Path('.')
sys.path.append(str(ROOT_DIR / 'src'))

from src.dataset import (
    build_cremad_audio_dataloaders,
    create_dataloaders_all,
)
from src.model_builder import (
    AudioBackboneCNN,
    AudioEmbeddingHead,
    BackboneWithClassifier,
    FrozenBackboneWithHead,
    build_quantum_head,
)
from src.training import (
    FineTuneConfig,
    train_with_history,
    pretrain_backbone_and_embedding,
    finetune_head_only,
    summarize_experiments,
    quantum_probability_helper,
)


In [ ]:

@dataclass
class ExperimentConfig:
    experiment_type: str  # 'embeddings', 'backbone_finetune', 'cnn_scratch'
    representation: str   # 'mel', 'mfcc', or 'embeddings'
    embedding_model: str = 'emb_resnet18'
    use_quantum: bool = False
    n_qubits: int = 10
    q_depth: int = 2
    pretrain_epochs: int = 4
    finetune_epochs: int = 10
    batch_size: int = 8
    learning_rate: float = 1e-3
    learning_rate_quantum: float = 1e-3
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:

# Dataset and artifact paths (adjust to your environment)
DATA_ROOT = Path('CREMAD')
AUDIO_DIR = DATA_ROOT / 'AudioWAV'
SPLITS_DIR = DATA_ROOT / 'splits'
EMBEDDING_DIR = DATA_ROOT / 'Embeddings'
MFCC_DIR = DATA_ROOT / 'MFCCs'
BACKBONE_DIR = DATA_ROOT / 'Models' / 'backbone'


In [ ]:


def resolve_device(name: str | None = None):
    if name:
        return torch.device(name)
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def build_embedding_head_model(class_names: List[str], cfg: ExperimentConfig, sample_batch: torch.Tensor) -> nn.Module:
    input_dim = sample_batch.shape[1]
    head_classical = nn.Linear(cfg.n_qubits, len(class_names))
    head_quantum = build_quantum_head(cfg.n_qubits, len(class_names), cfg.q_depth)
    feature_mapper = nn.Sequential(
        nn.Linear(input_dim, cfg.n_qubits),
        nn.ReLU(),
    )
    head = head_quantum if cfg.use_quantum else head_classical
    return nn.Sequential(feature_mapper, head)


def build_cnn_model(class_names: List[str], cfg: ExperimentConfig, pretrained: bool) -> nn.Module:
    backbone = AudioBackboneCNN(input_channels=1, pretrained=pretrained)
    embedding = AudioEmbeddingHead(backbone.output_dim, cfg.n_qubits, dropout=0.1)
    head = build_quantum_head(cfg.n_qubits, len(class_names), cfg.q_depth) if cfg.use_quantum else nn.Linear(cfg.n_qubits, len(class_names))
    return BackboneWithClassifier(backbone, embedding, head)


def load_embeddings_dataloaders(cfg: ExperimentConfig):
    config = {
        'base_model': cfg.embedding_model,
        'selected_classes': None,
        'batch_size': cfg.batch_size,
        'grayscale': False,
        'use_pretrained': True,
        'use_generic_weights': False,
        'embedding_dir': str(EMBEDDING_DIR),
        'specs_dir': str(DATA_ROOT / 'Spectrograms'),
        'mfcc_dir': str(MFCC_DIR),
    }
    dataloaders, dataset_sizes, class_names, _ = create_dataloaders_all(config, shuffle=True, num_workers=2)
    return dataloaders, dataset_sizes, class_names


def load_audio_dataloaders(cfg: ExperimentConfig):
    dataloaders, dataset_sizes, class_names = build_cremad_audio_dataloaders(
        audio_dir=str(AUDIO_DIR),
        splits_dir=str(SPLITS_DIR),
        representation=cfg.representation,
        batch_size=cfg.batch_size,
        num_workers=2,
    )
    return dataloaders, dataset_sizes, class_names


def run_embeddings_experiment(cfg: ExperimentConfig):
    dataloaders, dataset_sizes, class_names = load_embeddings_dataloaders(cfg)
    sample_batch, _ = next(iter(dataloaders['train']))
    model = build_embedding_head_model(class_names, cfg, sample_batch)
    device = resolve_device(cfg.device)
    model_dir = BACKBONE_DIR / 'embeddings' / ('quantum' if cfg.use_quantum else 'classical')
    model_dir.mkdir(parents=True, exist_ok=True)
    model, history = train_with_history(
        model,
        dataloaders,
        dataset_sizes,
        device=device,
        num_epochs=cfg.finetune_epochs,
        learning_rate=cfg.learning_rate_quantum if cfg.use_quantum else cfg.learning_rate,
        model_dir=str(model_dir),
        phases=tuple(phase for phase in ('train', 'val') if phase in dataloaders),
    )
    return {'head_type': 'quantum' if cfg.use_quantum else 'classical', 'representation': 'emb', 'n_qubits': cfg.n_qubits, 'q_depth': cfg.q_depth, 'final_acc': history.get('val_acc', [0])[-1] if history.get('val_acc') else 0}


def run_backbone_finetune(cfg: ExperimentConfig):
    dataloaders, dataset_sizes, class_names = load_audio_dataloaders(cfg)
    ft_cfg = FineTuneConfig(
        representation=cfg.representation,
        n_qubits=cfg.n_qubits,
        q_depth=cfg.q_depth,
        pretrain_epochs=cfg.pretrain_epochs,
        finetune_epochs=cfg.finetune_epochs,
        learning_rate_pretrain=cfg.learning_rate,
        learning_rate_finetune_classical=cfg.learning_rate,
        learning_rate_finetune_quantum=cfg.learning_rate_quantum,
        batch_size=cfg.batch_size,
        backbone_dir=str(BACKBONE_DIR),
        device_override=cfg.device,
    )
    tag = cfg.representation
    checkpoint_path = BACKBONE_DIR / f"{tag}_backbone.pt"
    if not checkpoint_path.exists():
        print('No backbone checkpoint found; running pretraining first...')
        pretrain_backbone_and_embedding(dataloaders, dataset_sizes, class_names, ft_cfg, representation_tag=tag)
    head_type = 'quantum' if cfg.use_quantum else 'classical'
    _, history, final_acc = finetune_head_only(
        checkpoint_path=str(checkpoint_path),
        dataloaders=dataloaders,
        dataset_sizes=dataset_sizes,
        class_names=class_names,
        config=ft_cfg,
        head_type=head_type,
        representation_tag=tag,
    )
    return {'head_type': head_type, 'representation': tag, 'n_qubits': cfg.n_qubits, 'q_depth': cfg.q_depth, 'final_acc': final_acc}


def run_cnn_scratch(cfg: ExperimentConfig):
    dataloaders, dataset_sizes, class_names = load_audio_dataloaders(cfg)
    model = build_cnn_model(class_names, cfg, pretrained=False)
    device = resolve_device(cfg.device)
    model_dir = DATA_ROOT / 'Models' / 'scratch' / cfg.representation / ('quantum' if cfg.use_quantum else 'classical')
    model_dir.mkdir(parents=True, exist_ok=True)
    model, history = train_with_history(
        model,
        dataloaders,
        dataset_sizes,
        device=device,
        num_epochs=cfg.finetune_epochs,
        learning_rate=cfg.learning_rate_quantum if cfg.use_quantum else cfg.learning_rate,
        model_dir=str(model_dir),
        phases=tuple(phase for phase in ('train', 'val') if phase in dataloaders),
    )
    eval_phase = 'val' if 'val_acc' in history else 'train'
    final_acc = history.get(f'{eval_phase}_acc', [0])[-1] if history.get(f'{eval_phase}_acc') else 0
    return {'head_type': 'quantum' if cfg.use_quantum else 'classical', 'representation': cfg.representation, 'n_qubits': cfg.n_qubits, 'q_depth': cfg.q_depth, 'final_acc': final_acc}


def run_experiment(cfg: ExperimentConfig):
    if cfg.experiment_type == 'embeddings':
        return run_embeddings_experiment(cfg)
    if cfg.experiment_type == 'backbone_finetune':
        return run_backbone_finetune(cfg)
    if cfg.experiment_type == 'cnn_scratch':
        return run_cnn_scratch(cfg)
    raise ValueError(f'Unsupported experiment type: {cfg.experiment_type}')


In [ ]:

# Widgets for interactive control
exp_type = Dropdown(options=[('Transfer on embeddings', 'embeddings'), ('Backbone fine-tuning (head only)', 'backbone_finetune'), ('CNN from scratch', 'cnn_scratch')], value='embeddings', description='Experiment')
representation = Dropdown(options=[('MEL', 'mel'), ('MFCC', 'mfcc'), ('Embeddings', 'embeddings')], value='mel', description='Input')
use_quantum = ToggleButtons(options=[('Classical', False), ('Quantum', True)], value=False, description='Head')
n_qubits = IntSlider(value=10, min=4, max=16, step=1, description='n_qubits')
q_depth = IntSlider(value=2, min=1, max=4, step=1, description='q_depth')
pretrain_epochs = IntSlider(value=4, min=1, max=20, step=1, description='Pretrain')
finetune_epochs = IntSlider(value=6, min=1, max=30, step=1, description='Finetune')
batch_size = IntSlider(value=8, min=2, max=32, step=2, description='Batch')
lr_classical = FloatText(value=1e-3, description='LR classic')
lr_quantum = FloatText(value=1e-3, description='LR quantum')
run_button = Button(description='Run experiment', button_style='success')
log_output = Output()

# Ensure representation matches experiment choice

def _sync_representation(*_):
    if exp_type.value == 'embeddings':
        representation.value = 'embeddings'
        representation.disabled = True
    else:
        if representation.value == 'embeddings':
            representation.value = 'mel'
        representation.disabled = False

exp_type.observe(_sync_representation, 'value')
_sync_representation()


def on_run_clicked(_):
    log_output.clear_output()
    with log_output:
        cfg = ExperimentConfig(
            experiment_type=exp_type.value,
            representation=representation.value,
            use_quantum=use_quantum.value,
            n_qubits=n_qubits.value,
            q_depth=q_depth.value,
            pretrain_epochs=pretrain_epochs.value,
            finetune_epochs=finetune_epochs.value,
            batch_size=batch_size.value,
            learning_rate=lr_classical.value,
            learning_rate_quantum=lr_quantum.value,
        )
        print('Running experiment with config:', cfg)
        try:
            result = run_experiment(cfg)
            summarize_experiments([result])
        except Exception as exc:  # pylint: disable=broad-except
            print('Experiment failed:', exc)

run_button.on_click(on_run_clicked)

controls_left = VBox([exp_type, representation, use_quantum, HBox([n_qubits, q_depth])])
controls_right = VBox([HBox([pretrain_epochs, finetune_epochs]), batch_size, lr_classical, lr_quantum, run_button])
display(VBox([HBox([controls_left, controls_right]), log_output]))


## Optional: quick smoke test

Run the cell below to load a tiny batch and verify the shapes without training the full models.


In [ ]:

# Smoke test: load a small batch for the selected representation
try:
    cfg = ExperimentConfig(experiment_type='cnn_scratch', representation=representation.value, use_quantum=False, batch_size=4)
    dls, _, _ = (load_embeddings_dataloaders(cfg) if representation.value == 'embeddings' else load_audio_dataloaders(cfg))
    phase = 'train' if 'train' in dls else next(iter(dls.keys()))
    xb, yb = next(iter(dls[phase]))
    print('Loaded batch shape:', xb.shape, 'labels shape:', yb.shape)
except Exception as exc:  # pylint: disable=broad-except
    print('Smoke test failed:', exc)
